#Initialization

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#Read Bronze table

In [0]:
df = spark.table("pcat.bronze.gross_price")

In [0]:
df.show(10)

#Silver Transformations

##Normalization

In [0]:
df = df.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
)

##Check Dataframe

In [0]:
df.select("month").distinct().show(10)

##Handling gross_price

In [0]:
df = df.withColumn(
    "gross_price",
    F.when(
        F.col("gross_price").rlike(r"^-?\d+(\.\d+)?$"),
        F.abs(F.col("gross_price").cast("double"))
    ).otherwise(F.lit(0.0))
)

In [0]:
df.show(10)

##Enrich The Table Data

In [0]:
product_df = spark.table("pcat.silver.products")
df = df.join(product_df.select("product_id", "product_code"), on = "product_id", how = "inner")

In [0]:
df = df.select("product_id", "product_code", "month", "gross_price", "read_timestamp", "file_name", "file_size")
df.show()

#Writing Silver Table

In [0]:
df.write \
  .format("delta") \
  .option("delta.enableChangeDataFeed", "true") \
  .option("mergeSchema", "true") \
  .mode("overwrite") \
  .saveAsTable("pcat.silver.gross_price")

##Sanity checks of silver _table_

In [0]:
%sql
SELECT * FROM pcat.silver.gross_price LIMIT 10